In [ ]:
!pip install -q mediapipe opencv-python-headless

In [ ]:
!git clone https://github.com/its-bhavya/workout_narration.git
!pip install mediapipe opencv-python-headless numpy matplotlib
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
for root, dirs, files in os.walk("data"):
    depth = root.count(os.sep)
    if depth <= 2:
        print(root, "->", files[:5], f"({len(files)} files)")

In [ ]:
import pandas as pd, json, os
m = "/content/workout_narration/data/manifests/"

filtered = pd.read_csv(m + "repcount_filtered.csv")
print(filtered.columns.tolist())
print(filtered.head())

splits = pd.read_csv(m + "splits.csv")
print(splits.columns.tolist())
print(splits.head())

with open(m + "repcount_rep_boundaries.json") as f:
    bounds = json.load(f)
k = list(bounds.keys())[0]
print(k, bounds[k])

In [ ]:
import pandas as pd
import os
df = pd.read_csv("/content/workout_narration/data/manifests/repcount_filtered.csv")
print(df.columns.tolist())
df.head()

In [ ]:
squats = df[df["exercise"] == "squat"]
print(squats.head())
video_name = squats.iloc[0]["filename"]
print(video_name)

In [ ]:
import os
VIDEO_PATH = None
for root, dirs, files in os.walk("/content"):
    if video_name in files:
        VIDEO_PATH = os.path.join(root, video_name)
        break
print(VIDEO_PATH)

In [ ]:
!find /content -name "*.mp4" | head -20

In [ ]:
import json
with open("/content/workout_narration/data/manifests/repcount_rep_boundaries.json") as f:
    bounds = json.load(f)

key = "stu2_25.mp4"           # using an existing key from the JSON to verify
print(list(bounds.keys())[:5])   # check how keys are written
print(bounds[key])
print("true reps:", len(bounds[key]))

In [ ]:
print(VIDEO_PATH)

In [ ]:
import glob
VIDEO_PATH = glob.glob("/content/drive/MyDrive/videos_exer/squats.mp4")[0]
print(VIDEO_PATH)

In [ ]:
# to check whats there in the video
import cv2, matplotlib.pyplot as plt
cap = cv2.VideoCapture(VIDEO_PATH)
print("fps:", cap.get(cv2.CAP_PROP_FPS),
      "frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
      "size:", int(cap.get(3)), "x", int(cap.get(4)))
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
ok, frame = cap.read()
cap.release()
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()

In [ ]:
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

In [ ]:
import cv2, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

options = vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO)
landmarker = vision.PoseLandmarker.create_from_options(options)

def extract_knee_angles(video_path):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    angles, i = [], 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        img = mp.Image(image_format=mp.ImageFormat.SRGB,
                       data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = landmarker.detect_for_video(img, int(i * 1000 / fps))
        if res.pose_landmarks:
            lm = res.pose_landmarks[0]
            angles.append(angle([lm[23].x, lm[23].y],
                                [lm[25].x, lm[25].y],
                                [lm[27].x, lm[27].y]))
        else:
            angles.append(np.nan)
        i += 1
    cap.release()
    return np.array(angles), fps

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

# Helper function to compute angle between 3 points
def angle(a, b, c):
    a = np.array(a)
    b = np.array(b) # Vertex
    c = np.array(c)

    radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - np.arctan2(a[1]-b[1], a[0]-b[0])
    angle_deg = np.abs(radians*180.0/np.pi)

    if angle_deg > 180.0:
        angle_deg = 360 - angle_deg

    return angle_deg

# 1. Extract knee angles using the previously defined function
print("Processing video... This may take a minute.")
angles, fps = extract_knee_angles(VIDEO_PATH)

# 2. Create the time array 't'
t = np.arange(len(angles)) / fps

# 3. Clean the angles and find peaks
clean = pd.Series(angles).interpolate().bfill().ffill().values
dips, _ = find_peaks(-clean, prominence=30)
print("dips found:", len(dips))

# 4. Plot the results
plt.figure(figsize=(10, 3))
plt.plot(t, clean, label="Knee Angle")
plt.plot(t[dips], clean[dips], "rx", label="Detected Reps")
plt.xlabel("time (s)")
plt.ylabel("knee angle (deg)")
plt.legend()
plt.show()

In [ ]:
from scipy.signal import savgol_filter

smooth = savgol_filter(clean, 9, 2)

def count_reps(sig, down_thresh, up_thresh):
    state, reps, rep_frames = "UP", 0, []
    for i, a in enumerate(sig):
        if np.isnan(a):
            continue
        if state == "UP" and a < down_thresh:
            state = "DOWN"
        elif state == "DOWN" and a > up_thresh:
            state = "UP"
            reps += 1
            rep_frames.append(i)
    return reps, rep_frames

DOWN_T, UP_T = 150, 168     # from your graph: bottoms <= ~145, tops ~175-180
reps, rep_frames = count_reps(smooth, DOWN_T, UP_T)
print("state machine reps:", reps, "| dips:", len(dips))

plt.figure(figsize=(10, 3))
plt.plot(t, smooth)
plt.axhline(DOWN_T, color="r", ls="--", label="down")
plt.axhline(UP_T, color="g", ls="--", label="up")
plt.plot(t[rep_frames], smooth[rep_frames], "ko", label="rep done")
plt.legend(); plt.show()

In [ ]:
import glob, cv2, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

def extract_angles(video_path, joints):
    a_i, b_i, c_i = joints
    opts = vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker.task"),
        running_mode=vision.RunningMode.VIDEO)
    lm_er = vision.PoseLandmarker.create_from_options(opts)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    angles, vis, i = [], [], 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        img = mp.Image(image_format=mp.ImageFormat.SRGB,
                       data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = lm_er.detect_for_video(img, int(i * 1000 / fps))
        if res.pose_landmarks:
            lm = res.pose_landmarks[0]
            angles.append(angle([lm[a_i].x, lm[a_i].y],
                                [lm[b_i].x, lm[b_i].y],
                                [lm[c_i].x, lm[c_i].y]))
            vis.append(min(lm[a_i].visibility, lm[b_i].visibility, lm[c_i].visibility))
        else:
            angles.append(np.nan); vis.append(0)
        i += 1
    cap.release()
    return np.array(angles), np.array(vis), fps

In [ ]:
PU_PATH = glob.glob("/content/drive/MyDrive/videos_exer/PushUps.mp4")[0]

left,  vL, fps = extract_angles(PU_PATH, (11, 13, 15))   # left shoulder, elbow, wrist
right, vR, _   = extract_angles(PU_PATH, (12, 14, 16))   # right side
print("mean visibility  left:", vL.mean().round(2), " right:", vR.mean().round(2))

pu = left if vL.mean() >= vR.mean() else right
tp = np.arange(len(pu)) / fps

In [ ]:
pu_clean = pd.Series(pu).interpolate(limit=5).bfill().ffill().values
pu_smooth = savgol_filter(pu_clean, 9, 2)

plt.figure(figsize=(10, 3))
plt.plot(tp, pu_smooth)
plt.xlabel("time (s)"); plt.ylabel("elbow angle (deg)")
plt.show()
print("max:", pu_smooth.max().round(), " min:", pu_smooth.min().round())

In [ ]:
DOWN_T, UP_T = 100, 150      # adjust from your plot
reps, rep_frames = count_reps(pu_smooth, DOWN_T, UP_T)
print("push-up reps:", reps)

plt.figure(figsize=(10, 3))
plt.plot(tp, pu_smooth)
plt.axhline(DOWN_T, color="r", ls="--"); plt.axhline(UP_T, color="g", ls="--")
plt.plot(tp[rep_frames], pu_smooth[rep_frames], "ko")
plt.show()

In [ ]:
#extracting both the arms
JJ_PATH = glob.glob("/content/drive/MyDrive/videos_exer/jumpingJacks.mp4")[0]

jl, vl, fps = extract_angles(JJ_PATH, (23, 11, 13))   # left hip, shoulder, elbow
jr, vr, _   = extract_angles(JJ_PATH, (24, 12, 14))   # right hip, shoulder, elbow
print("mean visibility  left:", vl.mean().round(2), " right:", vr.mean().round(2))

jj = np.nanmean([jl, jr], axis=0)      # average of both arms
tj = np.arange(len(jj)) / fps

In [ ]:
jj_clean = pd.Series(jj).interpolate(limit=5).bfill().ffill().values
jj_smooth = savgol_filter(jj_clean, 9, 2)

plt.figure(figsize=(10, 3))
plt.plot(tj, jj_smooth)
plt.xlabel("time (s)"); plt.ylabel("arm raise angle (deg)")
plt.show()
print("max:", jj_smooth.max().round(), " min:", jj_smooth.min().round())

In [ ]:
def count_cycles(sig, low_t, high_t):
    """Rep = signal goes below low_t, rises above high_t, returns below low_t."""
    state, reps, rep_frames = "LOW", 0, []
    for i, a in enumerate(sig):
        if np.isnan(a):
            continue
        if state == "LOW" and a > high_t:
            state = "HIGH"
        elif state == "HIGH" and a < low_t:
            state = "LOW"
            reps += 1
            rep_frames.append(i)
    return reps, rep_frames

LOW_T, HIGH_T = 60, 120      # adjust from your plot
reps, rep_frames = count_cycles(jj_smooth, LOW_T, HIGH_T)
print("jumping jack reps:", reps)

plt.figure(figsize=(10, 3))
plt.plot(tj, jj_smooth)
plt.axhline(LOW_T, color="r", ls="--"); plt.axhline(HIGH_T, color="g", ls="--")
plt.plot(tj[rep_frames], jj_smooth[rep_frames], "ko")
plt.show()

In [ ]:
import sys
sys.path.append("/content")
from counter import *
print(EXERCISES.keys())

In [ ]:
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task
!pip install -q mediapipe opencv-python-headless

In [ ]:
import os
print(os.listdir("/content/drive/MyDrive"))

In [ ]:
print(os.listdir("/content/drive/MyDrive/DATASET_REP"))

In [ ]:
import os
import tarfile

tar_path = "/content/drive/MyDrive/DATASET_REP/RepCountA (1).tar.gz"
extract_path = "/content/RepCountA"

# Extracting the dataset if not already extracted
if os.path.exists(tar_path):
    if not os.path.exists(extract_path):
        print("Extracting dataset... This might take a few moments.")
        with tarfile.open(tar_path, "r:gz") as tar:
            tar.extractall(path=extract_path)
        print("Extraction complete.")

    # List files in the extracted directory (checking common subdirectories like 'raw')
    possible_raw_dir = os.path.join(extract_path, "raw")
    if os.path.exists(possible_raw_dir):
        print("Contents of 'raw' directory:", os.listdir(possible_raw_dir)[:5])
    else:
        print("Root contents of extracted archive:", os.listdir(extract_path)[:5])
else:
    print(f"Tarball not found at {tar_path}. Please verify the path.")

In [ ]:
!find /content/RepCountA -name "*.mp4" | head -5
!find /content/RepCountA -name "*.mp4" | wc -l

In [ ]:
VIDEO_DIR = "/content/RepCountA/LLSP/video"

In [ ]:
import glob
def find_clip(name):
    hits = glob.glob(f"{VIDEO_DIR}/**/{name}", recursive=True)
    return hits[0] if hits else None

In [ ]:
import pandas as pd
df = pd.read_csv("/content/workout_narration/data/manifests/repcount_filtered.csv")
row = df[(df.exercise == "squat") & (df.split == "train")].iloc[0]
print(row.filename, "true reps:", row.rep_count)
path = find_clip(row.filename)
print(path)

In [ ]:
sig, fps = extract_exercise_angles(path, "squat")
out = count_for_exercise(clean_signal(sig), "squat", fps)
print("predicted:", out["reps"], " true:", row.rep_count)

In [ ]:
import ast, numpy as np, matplotlib.pyplot as plt

s = clean_signal(sig)
t = np.arange(len(s)) / fps
true_pairs = ast.literal_eval(row.rep_boundaries)      # [[start_frame, end_frame], ...]
d, u = EXERCISES["squat"]["thresholds"]

plt.figure(figsize=(11, 3.5))
plt.plot(t, s)
plt.axhline(d, color="r", ls="--", label="down thresh")
plt.axhline(u, color="g", ls="--", label="up thresh")
for a, b in true_pairs:
    plt.axvspan(a / fps, b / fps, color="orange", alpha=0.25)   # true reps
for rt in out["rep_times"]:
    plt.axvline(rt, color="k", lw=1)                            # your counted reps
plt.xlabel("time (s)"); plt.ylabel("knee angle (deg)")
plt.legend(); plt.show()
print("fps:", fps, "| true reps:", len(true_pairs), "| predicted:", out["reps"])

In [ ]:
squat_rows = df[(df.exercise == "squat") & (df.split == "train")].head(10)
signals, results = {}, []

for _, r in squat_rows.iterrows():
    p = find_clip(r.filename)
    if p is None:
        print("not found:", r.filename); continue
    sg, f = extract_exercise_angles(p, "squat")
    signals[r.filename] = (clean_signal(sg), f)
    o = count_for_exercise(signals[r.filename][0], "squat", f)
    results.append(dict(file=r.filename, true=r.rep_count, pred=o["reps"],
                        err=abs(o["reps"] - r.rep_count), fps=round(f, 1)))

res = pd.DataFrame(results)
print(res.to_string())
print("exact match:", (res.err == 0).mean(), "| within 1:", (res.err <= 1).mean(), "| MAE:", res.err.mean())

In [ ]:
#problem with video stud_62
"""import ast, numpy as np, matplotlib.pyplot as plt

name = "stu6_62.mp4"
s, f = signals[name]
r = df[df.filename == name].iloc[0]
tt = np.arange(len(s)) / f
d, u = EXERCISES["squat"]["thresholds"]

plt.figure(figsize=(11, 3.5))
plt.plot(tt, s)
plt.axhline(d, color="r", ls="--"); plt.axhline(u, color="g", ls="--")
for a, b in ast.literal_eval(r.rep_boundaries):
    plt.axvspan(a / f, b / f, color="orange", alpha=0.25)
plt.xlabel("time (s)"); plt.ylabel("knee angle (deg)"); plt.show()
print("min:", np.nanmin(s).round(), " max:", np.nanmax(s).round(),
      " NaN share:", np.isnan(s).mean().round(2))"""

In [ ]:
#from IPython.display import Video
#Video(find_clip(name), embed=True, width=400)

In [ ]:
def extract_angles_multi(video_path, joints, model_path="pose_landmarker.task", num_poses=3):
    import cv2, mediapipe as mp
    from mediapipe.tasks import python as mp_python
    from mediapipe.tasks.python import vision

    a_i, b_i, c_i = joints
    opts = vision.PoseLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=model_path),
        running_mode=vision.RunningMode.VIDEO, num_poses=num_poses)
    lmk = vision.PoseLandmarker.create_from_options(opts)

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    tracks = []                      # each: {"center": (x, y), "angles": [..]}
    n = 0
    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break
        img = mp.Image(image_format=mp.ImageFormat.SRGB,
                       data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        res = lmk.detect_for_video(img, int(n * 1000 / fps))

        dets = []
        for lm in res.pose_landmarks:
            center = ((lm[23].x + lm[24].x) / 2, (lm[23].y + lm[24].y) / 2)
            ang = angle([lm[a_i].x, lm[a_i].y], [lm[b_i].x, lm[b_i].y], [lm[c_i].x, lm[c_i].y])
            dets.append((center, ang))

        used = set()
        for tr in tracks:            # match each existing track to the nearest detection
            best, best_d = None, 0.15
            for j, (c, _) in enumerate(dets):
                if j in used:
                    continue
                dist = ((c[0] - tr["center"][0]) ** 2 + (c[1] - tr["center"][1]) ** 2) ** 0.5
                if dist < best_d:
                    best, best_d = j, dist
            if best is None:
                tr["angles"].append(np.nan)
            else:
                used.add(best)
                tr["center"] = dets[best][0]
                tr["angles"].append(dets[best][1])
        for j, (c, a) in enumerate(dets):      # new person -> new track
            if j not in used:
                tracks.append({"center": c, "angles": [np.nan] * n + [a]})
        for tr in tracks:                       # keep all tracks the same length
            if len(tr["angles"]) < n + 1:
                tr["angles"].append(np.nan)
        n += 1
    cap.release()

    # choose the track that moves the most (the exerciser)
    best_sig, best_score = np.full(n, np.nan), -1
    for tr in tracks:
        arr = np.array(tr["angles"])
        if np.isfinite(arr).mean() < 0.2:
            continue
        score = np.nanstd(arr)
        if score > best_score:
            best_sig, best_score = arr, score
    return best_sig, fps

In [ ]:
sg, f = extract_angles_multi(find_clip("stu6_62.mp4"), EXERCISES["squat"]["left"])
s2 = clean_signal(sg)
print(count_for_exercise(s2, "squat", f), "| true: 20")

plt.figure(figsize=(11, 3)); plt.plot(np.arange(len(s2)) / f, s2); plt.show()

In [ ]:
opts = vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="pose_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO, num_poses=3,
    min_pose_detection_confidence=0.3,
    min_pose_presence_confidence=0.3,
    min_tracking_confidence=0.3)

In [ ]:
best, best_d = None, 0.4      # was 0.15

In [ ]:
sg, f = extract_angles_multi(find_clip("stu6_62.mp4"), EXERCISES["squat"]["left"])
s2 = clean_signal(sg)
print(count_for_exercise(s2, "squat", f), "| true: 20")

plt.figure(figsize=(11, 3)); plt.plot(np.arange(len(s2)) / f, s2); plt.show()

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
yolo = YOLO("yolov8m-pose.pt")        # the "m" model is better at small people than "n"

def extract_angles_yolo(video_path, joints=(11, 13, 15)):   # COCO: left hip, knee, ankle
    import cv2
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    tracks = {}
    for i, r in enumerate(yolo.track(video_path, persist=True, stream=True, verbose=False)):
        if r.boxes.id is None:
            continue
        ids = r.boxes.id.int().cpu().tolist()
        kp = r.keypoints.xy.cpu().numpy()
        cf = r.keypoints.conf.cpu().numpy()
        for pid, k, c in zip(ids, kp, cf):
            a, b, cc = joints
            if min(c[a], c[b], c[cc]) < 0.3:
                continue
            tracks.setdefault(pid, np.full(n, np.nan))[i] = angle(k[a], k[b], k[cc])
    best = max(tracks.values(), key=lambda x: np.nanstd(x) if np.isfinite(x).sum() > 15 else -1)
    return best, fps

In [ ]:
sg, f = extract_angles_yolo(find_clip("stu6_62.mp4"))
s2 = clean_signal(sg)
print(count_for_exercise(s2, "squat", f)["reps"], "| true: 20")

In [ ]:
def run_batch(exercise, n=10):
    rows = df[(df.exercise == exercise) & (df.split == "train")].head(n)
    out_rows = []
    for _, r in rows.iterrows():
        p = find_clip(r.filename)
        if p is None:
            continue
        sg, f = extract_exercise_angles(p, exercise)
        o = count_for_exercise(clean_signal(sg), exercise, f)
        out_rows.append(dict(file=r.filename, true=r.rep_count, pred=o["reps"],
                             err=abs(o["reps"] - r.rep_count)))
    res = pd.DataFrame(out_rows)
    print(exercise, "| exact:", (res.err == 0).mean(),
          "| within 1:", (res.err <= 1).mean(), "| MAE:", round(res.err.mean(), 2))
    return res

res_pu = run_batch("push_up")
print(res_pu.to_string())

In [ ]:
def run_batch(exercise, n=10):
    rows = df[(df.exercise == exercise) & (df.split == "train")].head(n)
    out_rows = []
    for _, r in rows.iterrows():
        p = find_clip(r.filename)
        if p is None:
            continue
        sg, f = extract_exercise_angles(p, exercise)
        o = count_for_exercise(clean_signal(sg), exercise, f)
        out_rows.append(dict(file=r.filename, true=r.rep_count, pred=o["reps"],
                             err=abs(o["reps"] - r.rep_count)))
    res = pd.DataFrame(out_rows)
    print(exercise, "| exact:", (res.err == 0).mean(),
          "| within 1:", (res.err <= 1).mean(), "| MAE:", round(res.err.mean(), 2))
    return res

res_jj = run_batch("jump_jack")
print(res_jj.to_string())

In [ ]:
name = "stu7_49.mp4"
r = df[df.filename == name].iloc[0]
sg, f = extract_exercise_angles(find_clip(name), "push_up")
s = clean_signal(sg)
d, u = EXERCISES["push_up"]["thresholds"]

import ast
plt.figure(figsize=(12, 3.5))
plt.plot(np.arange(len(s)) / f, s)
plt.axhline(d, color="r", ls="--"); plt.axhline(u, color="g", ls="--")
for a, b in ast.literal_eval(r.rep_boundaries):
    plt.axvspan(a / f, b / f, color="orange", alpha=0.2)
plt.show()
print("5th/95th percentile:", np.nanpercentile(s, 5).round(), np.nanpercentile(s, 95).round(),
      "| NaN share:", np.isnan(s).mean().round(2))

In [ ]:
EXERCISES["push_up"]["thresholds"] = (135, 148)    # (down, up)
print(count_for_exercise(s, "push_up", f)["reps"], "| true:", r.rep_count)

In [ ]:
res_pu2 = run_batch("push_up")
print(res_pu2.to_string())

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
CACHE = "/content/drive/MyDrive/signals"
os.makedirs(CACHE, exist_ok=True)

def get_signal(filename, exercise):
    p = f"{CACHE}/{exercise}_{filename}.npz"
    if os.path.exists(p):
        z = np.load(p); return z["s"], float(z["fps"])
    sg, f = extract_exercise_angles(find_clip(filename), exercise)
    np.savez(p, s=sg, fps=f)
    return sg, f

def adaptive_thresholds(sig, low_frac=0.35, high_frac=0.65):
    lo, hi = np.nanpercentile(sig, 10), np.nanpercentile(sig, 90)
    return lo + low_frac * (hi - lo), lo + high_frac * (hi - lo)

def drop_close(times, min_gap=0.5):
    kept = []
    for t in times:
        if not kept or t - kept[-1] >= min_gap:
            kept.append(t)
    return kept

def run_batch(exercise, n=10, adaptive=True, low=0.35, high=0.65, min_gap=0.5, split="train"):
    rows = df[(df.exercise == exercise) & (df.split == split)].head(n)
    out_rows = []
    for _, r in rows.iterrows():
        sg, f = get_signal(r.filename, exercise)
        s = clean_signal(sg)
        th = adaptive_thresholds(s, low, high) if adaptive else None
        o = count_for_exercise(s, exercise, f, thresholds=th)
        pred = len(drop_close(o["rep_times"], min_gap))
        out_rows.append(dict(file=r.filename, true=r.rep_count, pred=pred,
                             err=abs(pred - r.rep_count)))
    res = pd.DataFrame(out_rows)
    print(exercise, "| exact:", (res.err == 0).mean(),
          "| within 1:", (res.err <= 1).mean(), "| MAE:", round(res.err.mean(), 2))
    return res

res_pu3 = run_batch("push_up")
print(res_pu3.to_string())

In [ ]:
for lo, hi in [(0.30, 0.70), (0.25, 0.75), (0.35, 0.65)]:
    print(lo, hi); run_batch("push_up", low=lo, high=hi)

In [ ]:
res_sq = run_batch("squat")
print(res_sq.to_string())

In [ ]:
res_jj = run_batch("jump_jack")
print(res_pu3.to_string())